# TP 05-A — Spam Detection with RNN and LSTM

**Course:** Natural Language Processing  
**Session:** 5 / 8  
**Submission deadline:** one week after the session

---

## Context

In Sessions 01–04 we represented documents as **fixed-size vectors** — either sparse (TF-IDF) or dense (GloVe mean pool). Both approaches discard word order entirely.

Today we process text as a **sequence**. A Recurrent Neural Network reads one token at a time, left to right, maintaining a hidden state that summarises everything seen so far. Word order finally matters.

**Task:** binary classification — is an SMS message spam or ham?

**Dataset:** 293 SMS messages (190 ham, 103 spam) embedded directly in this notebook. No download required.

---

## Roadmap

| Part | Task | New concept |
|------|------|-------------|
| 1 | Explore and preprocess the dataset | Class imbalance |
| 2 | Build the text pipeline: tokenise → pad | `Tokenizer`, `pad_sequences` |
| 3 | SimpleRNN classifier | `Embedding` + `SimpleRNN` |
| 4 | LSTM classifier | `LSTM`, gates, long-range memory |
| 5 | Compare and analyse | Training curves, error analysis |
| 6 (bonus) | Add a `Dense` baseline | Does word order actually help? |

---


## Setup


In [ ]:
# Uncomment to install on Colab
# !pip install tensorflow scikit-learn matplotlib -q

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Hyperparameters (modify for experiments) ──────────────────────────────────
MAX_VOCAB   = 1_000   # vocabulary size
MAX_LEN     = 50      # sequence length after padding
EMBED_DIM   = 16      # embedding dimension inside the network
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-3

print(f'TensorFlow version: {tf.__version__}')
print('Setup OK.')


---
## Part 1 — Dataset

The dataset is embedded below. No download required.
- **0 = ham** (legitimate message)
- **1 = spam**


In [ ]:
HAM_MESSAGES = [
    "Ok lar... Joking wif u oni...",
    "Fine if thats the way u feel. Thats the way its gota b",
    "Is that seriously how you spell his name?",
    "I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k?",
    "Oops, I'll let you know when my roommate's done.",
    "Ok",
    "Ill be there by 11",
    "Did you catch the game last night? Unreal finish.",
    "Can you pick up some milk on your way home?",
    "Meeting pushed to 3pm, same room",
    "Thanks for dinner, it was really good",
    "Running 10 mins late, save me a seat",
    "Happy birthday! Hope you have a great day",
    "Just got off the train, be there in 20",
    "The report is on your desk",
    "Are you free this weekend?",
    "Call me when you land",
    "Don't forget the dentist at 2",
    "Sounds good, see you then",
    "Let me know if you need anything",
    "On my way, parking now",
    "Can we reschedule to Friday?",
    "Left my charger at yours, can I grab it tomorrow?",
    "Dinner at 7 works for me",
    "The keys are under the mat",
    "Just finished the meeting, heading back now",
    "Can you send me that file again? I deleted it by mistake",
    "Sure, I'll be there at 6",
    "Pick you up at 8?",
    "Your parcel was delivered to the front desk",
    "Let's grab coffee after the lecture",
    "I'm at the library, come find me",
    "Quiz result was fine, don't stress",
    "The lecture slides are on the portal",
    "Can you cover my shift on Saturday?",
    "Just sent you the invoice",
    "We're out of coffee, can you grab some?",
    "Traffic is bad, might be 15 mins late",
    "Call me when you're free",
    "Already booked the table for 4",
    "Don't worry about it, happens to everyone",
    "The wifi password is Sunshine42",
    "See you at the station at half past",
    "No worries, take your time",
    "Done! I'll ping you the details",
    "Sorry for the late reply, been busy",
    "That works, thanks",
    "Just confirmed with the others, we're all coming",
    "Got your message, will call you tonight",
    "I think I left my bag in your car",
    "Can you print this for me?",
    "The gym opens at 6am",
    "My flight lands at 10:30",
    "Just checking in, how's the project going?",
    "The meeting notes are in the shared folder",
    "I need help moving this Saturday, any chance?",
    "Finished the essay, finally!",
    "Are you going to the seminar tomorrow?",
    "Let me know when you're done and I'll come get you",
    "That's fine with me",
    "I'm downstairs whenever you're ready",
    "Forgot my umbrella, hope it doesn't rain",
    "The doctor said everything looks fine",
    "Got the tickets, we're all set",
    "I'll bring the food if you bring the drinks",
    "Happy to help, just let me know when",
    "The car is at gate B7",
    "Yeah that sounds like a plan",
    "Can't talk right now, I'll call back later",
    "Leaving now, should be there by 7",
    "Did you remember to feed the cat?",
    "The presentation is at 10am sharp",
    "Already sent it, check your inbox",
    "No problem, take care",
    "See you tomorrow",
    "Let's sort this out when I get back",
    "Lost my phone for a bit, that's why I didn't reply",
    "The booking reference is PX2847",
    "I'll handle it, don't stress",
    "Just had a look, the document looks good to me",
    "At the checkout, won't be long",
    "Remind me tomorrow and I'll sort it",
    "That's a good point, hadn't thought of that",
    "The train is delayed by 20 minutes",
    "Sent you the link, let me know what you think",
    "I'm at home if you want to pop round",
    "Going to bed, chat tomorrow",
    "Can you check if I left my keys there?",
    "All good here, how about you?",
    "The seminar has been cancelled",
    "I'll call you when I'm on my way",
    "We need to talk about the project timeline",
    "Found a great restaurant, should we try it?",
    "There's a staff meeting at 2",
    "Thanks for the help earlier",
    "Glad it worked out!",
    "Just arrived, where are you?",
    "Need to cancel tonight, really sorry",
    "The new schedule is posted on the board",
    "I'm in the office until 6 if you need me",
    "All done, heading home now",
    "See you on the other side!",
    "Picked up the prescription for you",
    "Let me know your address and I'll drop it off",
    "The power went out for a bit, back now",
    "Almost finished, give me 10 more minutes",
    "Happy to chat whenever you're free",
    "Did you get my voicemail?",
    "The bus comes every 15 minutes",
    "Can you grab some bread on the way?",
    "Just woke up, running a bit behind",
    "Great, see you at the usual spot",
    "I think we should talk before making any decisions",
    "All booked! Confirmation sent to your email",
    "The exam results are out",
    "It was good seeing you today",
    "Let's catch up properly soon",
    "The password is Case-sensitive",
    "I'll forward it to you right away",
    "Just checking you got home okay",
    "The office is closed tomorrow",
    "I'll cover the cost, sort me out later",
    "Stuck in traffic, parking is a nightmare today",
    "Just a heads up, the system will be down from 2 to 4",
    "Tried calling but it went straight to voicemail",
    "Take an umbrella, it looks like rain",
    "The deposit has been transferred",
    "Nothing urgent, just checking in",
    "On a call, will text you when I'm free",
    "I left a note on your desk",
    "That sounds great, I'm in",
    "Do you have the number for the plumber?",
    "Already sorted, no need to worry",
    "You left your jacket here",
    "The report has been submitted",
    "Can we push the call to 4pm?",
    "All confirmed, see you Friday",
    "I owe you a coffee for this",
    "The code is 4821",
    "Almost there, about 5 minutes away",
    "Just got the email, thanks for sending it",
    "I'll let you know once I hear back",
    "No signal on the underground, just got your message",
    "Hope you feel better soon",
    "The spare key is with next door",
    "Running on time for once!",
    "Checked the calendar, that date works",
    "I'll read it tonight and get back to you",
    "The flight has been delayed by an hour",
    "Left you a voicemail, give me a ring when you can",
    "That's really kind, thank you",
    "The file is too large to send by email, using the shared drive",
    "Just sent the invite, check your calendar",
    "Are you coming to the team lunch on Thursday?",
    "I should be done by 4, then I'm all yours",
    "The network is slow today, bear with me",
    "Can't make it to the gym today, too much on",
    "Already on it",
    "See you at the usual place",
    "The bus was packed, needed to wait for the next one",
    "Let me look into it and get back to you",
    "What time does it start?",
    "The feedback was mostly positive",
    "I'll ask and let you know",
    "Heading to the shops, need anything?",
    "Good luck with the presentation!",
    "Just send it over whenever you're ready",
    "The meeting went well",
    "My battery is almost dead",
    "I think we're on the wrong floor",
    "Ready when you are",
    "I'll check and confirm by end of day",
    "Arrived safely, thanks for asking",
    "The room is booked from 10 to 12",
    "Could you ask him to call me?",
    "I put it in the fridge for you",
    "The map is shared in the group chat",
    "That's exactly what I needed, thanks",
    "I'll pick up the slack while you're away",
    "Just got out of the cinema, was brilliant",
    "Are we still on for Sunday?",
    "The handover notes are done",
    "I might be a few minutes early",
    "Let me check my diary and come back to you",
    "Your appointment is confirmed for Thursday at 11",
    "I've added it to the shared calendar",
    "All sorted, enjoy your evening",
    "Can't talk now but everything's fine",
    "The offer letter has been sent to your email",
    "Just picking up the kids, back in 30"
]

SPAM_MESSAGES = [
    "WINNER!! As a valued network customer you have been selected to receive a \u00a3900 prize reward! To claim call 09061701461.",
    "You have 1 new voicemail. Call 08719181513 to listen to it.",
    "URGENT! You have won a 1 week FREE membership in our \u00a3100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010",
    "Had your mobile 11 months or more? You are entitled to update to the latest colour mobiles with camera for FREE!",
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question.",
    "WINNER! Congratulations ur awarded 500 of CD vouchers or 125gift guaranteed & Free entry 2 100 wkly draw txt MUSIC to 87066",
    "You are a winner! Our records show you have been selected for a \u00a32000 cash prize. Call 0871 872 9815 NOW",
    "IMPORTANT - You could be entitled to up to \u00a33,160 in compensation from mis-sold PPI on a credit card or loan. Please reply PPI for info or STOP to opt out.",
    "Did you hear about the new lottery? You are pre-selected. Text LUCKY to 82277 to claim \u00a3500",
    "Your phone account is suspended. Call 0800 561 5566 immediately to reactivate.",
    "Your Amazon order cannot be delivered. Update your details here to avoid return.",
    "Congratulations! You have been chosen to receive a FREE iPhone 15. Tap here to claim before midnight.",
    "FINAL NOTICE: You owe an unpaid toll fee. Avoid late charges, pay now at our website.",
    "Your Netflix account has been suspended. Verify billing info now.",
    "We have been trying to reach you about your car extended warranty. Call 0800 111 2222 now.",
    "Claim your \u00a3200 Tesco voucher! You are one of today lucky winners. Reply YES to receive.",
    "FREE ringtones for life! Text RING to 86688. Only \u00a31.50 per week. Opt out: text STOP",
    "Urgent: Your bank account has been locked. Confirm identity at our secure site.",
    "You have a pending payment of \u00a36.90. Please update card details via the Post Office website.",
    "ALERT: Suspicious login detected on your account. Verify now at our secure link.",
    "PRIVATE! Are you over 18? This service is for adults only. 150p per msg. Reply ADULT to 69911.",
    "Your application has been approved! Claim your \u00a35000 loan now. No credit check. Call 08001 234 567",
    "Win a brand new Range Rover! Text WIN to 75555. Ts and Cs apply. \u00a32.50 per week subscription.",
    "Your PayPal account shows unusual activity. Login to our secure site now.",
    "You are selected for a special offer! Get 50 percent off today only. Click the link.",
    "SIX chances to win CASH! From \u00a3100 to \u00a320,000. To play, txt CUP to 87151. Cost 10p",
    "Todays Voda numbers ending with 7634 are selected to receive a \u00a3350 award.",
    "Customer service: your application for a \u00a32000 loan was approved. Call 0870 888 8888 now.",
    "Urgent! You have a delivery waiting. Confirm your address at our tracking site.",
    "This message is FREE. Get 50 free texts a week when you join TextMe Plus. Text PLUS to 11111",
    "You have been selected for our research panel. Earn \u00a350 for a 10 min survey. Text SURVEY to 60777",
    "Congratulations to you! Win a free holiday in our summer draw. Enter now on our website.",
    "APPLE ANNOUNCEMENT: You have won an iPhone 15 Pro in our monthly giveaway. To redeem visit our site.",
    "Your HSBC mobile banking PIN has been locked. Call us immediately on 03458 404 404",
    "You qualify for a refund of \u00a3312. Click here to claim your HMRC tax refund.",
    "Free msg: 3 mobile customers win \u00a31000 cash daily. Text CASH3 to 89097.",
    "REMINDER: Your 3 free months of premium service end TODAY. Renew or lose access.",
    "Exclusive invite: 500 BONUS credits added to your account! Sign in now.",
    "QUIZ MASTER: 3 questions. \u00a3500 prize. Text QUIZ to 45567. 25p per text. 18 plus only.",
    "Your delivery has failed due to incomplete address. Reschedule at our delivery site.",
    "You have been pre-selected for a Barclaycard with 0 percent interest. Apply in 2 mins.",
    "Mobile subscriber of the week! You have won a \u00a3250 Amazon voucher. Text AMAZON to 66600",
    "IMPORTANT INFO: Your council tax may be wrong. Claim a refund today at our website.",
    "Special deal today only! Get 3 months of BroadbandPlus for FREE. Reply INFO to 55544",
    "This is your final reminder to claim your PPI refund. You may be owed \u00a34,000 plus. Call 08007 000999",
    "WINNER again! You have been drawn as our lucky subscriber. \u00a3100 mobile credit. Reply WIN",
    "Your McDelivery order could not be processed. Re-enter payment at our site.",
    "Act now! Claim your personal injury compensation of up to \u00a32800. No win no fee. Text CLAIM to 88600",
    "VIP offer: 500 free spins and \u00a350 bonus at SpinCasino. 18 plus. Ts and Cs. Opt out: STOP",
    "Dear user, your account will expire. Click here to verify your details.",
    "TEXT WIN to 80082 for a chance to win \u00a31000 weekly. Cost: 25p per week.",
    "National Lottery Alert: A ticket matching your details has won \u00a31,450. Check our site.",
    "HMRC notification: A tax return has been filed in your name. Query this at our website.",
    "Your BT broadband is due for upgrade. Claim your FREE booster router. Reply YES to confirm",
    "Exclusive: 20,000 Avios points for switching energy supplier today. Text AVIOS to 59059",
    "Alert: Missed CCTV fine of \u00a365 on 14 April. Avoid court summons at our site.",
    "Orange customer, you are a WINNER of our monthly draw! \u00a3350 Argos voucher. Call 09064024321",
    "Our latest competition winner is YOU! \u00a3500 M and S voucher. Tap to claim.",
    "Reply YES to claim your FREE upgrade to unlimited texts and minutes on your current plan.",
    "O2 SPECIAL: Upgrade your handset for free this month only. Qualify at our website.",
    "Auto-reminder: Your vehicle licence disc expires soon. Renew online at our site.",
    "Hi there! We noticed you have not completed your profile. Get 100 bonus points.",
    "FREE CALL: You have 3 missed calls from a secret admirer. To reveal, text SECRET to 80155.",
    "Congratulations! McDonald is selected you for a paid survey. Earn \u00a375 today.",
    "We are unable to process your latest water bill payment. Update card at our site.",
    "Your Royal Mail parcel has 1 pending action. Visit our site before expiry.",
    "Want a \u00a3200 Primark gift card? Just answer 3 questions on our site!",
    "FREE: 500 texts per week on your current plan. Accept this offer at our site.",
    "Important: Your DWP Universal Credit requires urgent action. Log in at our site.",
    "Spend less on energy! Get a FREE quote today and save up to \u00a3600 per year. Text SAVE to 66655",
    "You have been shortlisted for a job role paying \u00a360,000. Apply now.",
    "Your subscription to FilmBox Premium has been auto-renewed for \u00a39.99. To cancel visit our site.",
    "FINAL WARNING: Your court hearing date has been set. Act now to avoid arrest.",
    "Claim your \u00a3500 Boots voucher now! Selected for you based on recent purchases.",
    "Lottery prize pending: \u00a37,500 awaiting collection. Visit our site before it expires.",
    "Warning: Your social media account will be disabled. Confirm identity at our site.",
    "Limited offer: Get 6 months FREE on our premium TV package. Text TV to 61001",
    "CASH: Congratulations! Text CASH to 22722 to claim your \u00a3200 instant cash prize.",
    "Your insurance renewal is overdue. Get a cheap quote at our site.",
    "Hi! You qualify for a council hardship grant of up to \u00a31,200. Apply at our site.",
    "Missed a delivery from Amazon? Rearrange here at our site.",
    "PRIZE NOTIFICATION: You won an all-inclusive trip to Barbados! Claim at our site.",
    "Your NatWest online banking has been disabled. Reactivate now at our site.",
    "Reminder: You have unclaimed refund of \u00a367.80 from DVLA. Claim at our site.",
    "This number won \u00a31,500 in our monthly prize draw. Call 08718726920 within 48 hours.",
    "Work from home and earn \u00a3300 a day! No experience needed. Start at our site.",
    "FREE broadband for 6 months for customers switching today.",
    "Earn cashback on every purchase! Apply for the best card.",
    "Alert! Your electricity direct debit failed. Update payment at our site.",
    "You have a benefits check due. Claim up to \u00a31,800 at our site.",
    "Exclusive mobile deal! Get 100GB data for just \u00a35 per month. Limited time.",
    "Your smart meter needs a software update. Book a FREE engineer visit.",
    "WINNER: Reply to claim your \u00a3500 Just Eat voucher selected randomly from UK numbers.",
    "Parking Charge Notice: \u00a360 unpaid. Avoid enforcement, pay at our site.",
    "Hi! We found your CV on Indeed. Top employers want to interview you. Register now.",
    "Your URGENT update is available. Tap to install and secure your device.",
    "Last chance! \u00a325,000 giveaway. 3 days left to enter.",
    "Your car tax disc expired 3 days ago. Renew now to avoid fine."
]

texts  = HAM_MESSAGES + SPAM_MESSAGES
labels = [0] * len(HAM_MESSAGES) + [1] * len(SPAM_MESSAGES)

df = pd.DataFrame({'text': texts, 'label': labels})
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Total messages : {len(df)}')
print(f'Ham  (label 0) : {(df.label == 0).sum()}')
print(f'Spam (label 1) : {(df.label == 1).sum()}')
print()
print('Sample messages:')
for _, row in df.sample(4, random_state=0).iterrows():
    label_str = 'HAM ' if row.label == 0 else 'SPAM'
    print(f'  [{label_str}] {row.text[:70]}')


### 1.1 — Explore message length and class balance


In [ ]:
def plot_length_distribution(
    df: pd.DataFrame,
    max_len: int,
) -> plt.Figure:
    """
    Plot the distribution of message lengths (in words) per class.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns 'text' and 'label'.
    max_len : int
        Sequence truncation length to mark on the plot.

    Returns
    -------
    matplotlib.figure.Figure
    """
    df = df.copy()
    df['n_words'] = df['text'].str.split().str.len()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for ax, (label_val, label_name, color) in zip(
        axes,
        [(0, 'Ham', '#56c8ba'), (1, 'Spam', '#f47067')]
    ):
        subset = df[df['label'] == label_val]['n_words']
        ax.hist(subset, bins=20, color=color, alpha=0.8, edgecolor='none')
        ax.axvline(max_len, color='white', linestyle='--', linewidth=1.2,
                   label=f'MAX_LEN={max_len}')
        ax.set_title(f'{label_name} message lengths', color='#e6edf3')
        ax.set_xlabel('Words per message')
        ax.set_ylabel('Count')
        ax.legend()
        ax.set_facecolor('#161b22')

    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.tick_params(colors='#8b949e')
        ax.spines[:].set_color('#30363d')
    plt.tight_layout()
    return fig


fig = plot_length_distribution(df, MAX_LEN)
plt.show()

# Word count stats
df['n_words'] = df['text'].str.split().str.len()
print('Message length (words):')
print(df.groupby('label')['n_words'].describe().round(1))


**📝 Your analysis — Part 1**

1. Are spam messages longer or shorter than ham on average? Does this make intuitive sense?
2. What fraction of messages are truncated by `MAX_LEN=50`? Does this seem like a problem?
3. The dataset has ~190 ham and ~103 spam. This is **class imbalance**. How might this affect the classifier's behaviour?

*Write your answers here.*


---
## Part 2 — Text Pipeline: Tokenise and Pad

A recurrent network needs integer sequences of equal length:

```
raw text  →  token list  →  integer sequence  →  padded matrix
"Call me now"  →  ['call','me','now']  →  [42, 8, 19]  →  [0, 0, 0, 42, 8, 19]
```

Keras `Tokenizer` handles steps 2 and 3. Padding (`pad_sequences`) handles step 4.


### 2.1 — Implement `build_sequences`


In [ ]:
def build_sequences(
    texts: list[str],
    tokenizer: Tokenizer | None = None,
    max_vocab: int = 1_000,
    max_len: int = 50,
    fit: bool = True,
) -> tuple[np.ndarray, Tokenizer]:
    """
    Convert a list of raw texts into a padded integer sequence matrix.

    Steps:
        1. If fit=True, fit a new Tokenizer on the texts.
           If fit=False, use the provided tokenizer (for val/test).
        2. Convert each text to a sequence of word indices.
        3. Pad/truncate all sequences to max_len
           (pre-padding with zeros, post-truncation).

    Parameters
    ----------
    texts : list of str
        Raw text documents.
    tokenizer : Tokenizer or None
        Pre-fitted tokenizer. Required when fit=False.
    max_vocab : int, optional
        Maximum vocabulary size, by default 1_000.
    max_len : int, optional
        Sequence length after padding, by default 50.
    fit : bool, optional
        Whether to fit the tokenizer on these texts, by default True.

    Returns
    -------
    sequences : np.ndarray, shape (len(texts), max_len)
        Padded integer sequence matrix.
    tokenizer : Tokenizer
        The fitted tokenizer (same object if fit=False).

    Raises
    ------
    ValueError
        If fit=False and no tokenizer is provided.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Train / val / test split (70 / 15 / 15), stratified
X_raw = df['text'].tolist()
y     = df['label'].values

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

# Fit tokenizer on train only
X_train, tokenizer = build_sequences(
    X_train_raw, max_vocab=MAX_VOCAB, max_len=MAX_LEN, fit=True
)
X_val,  _ = build_sequences(
    X_val_raw, tokenizer=tokenizer, max_len=MAX_LEN, fit=False
)
X_test, _ = build_sequences(
    X_test_raw, tokenizer=tokenizer, max_len=MAX_LEN, fit=False
)

print(f'Train : {X_train.shape}  labels: {Counter(y_train)}')
print(f'Val   : {X_val.shape}  labels: {Counter(y_val)}')
print(f'Test  : {X_test.shape}  labels: {Counter(y_test)}')
print(f'Vocab size (effective): {len(tokenizer.word_index)}')
print()
print('Example — original text:')
print(' ', X_train_raw[0])
print('Padded sequence (first 20 values):')
print(' ', X_train[0, -20:])


---
## Part 3 — SimpleRNN Classifier

Architecture:
```
Input (MAX_LEN,) → Embedding(MAX_VOCAB, EMBED_DIM) → SimpleRNN(units) → Dense(1, sigmoid)
```

The `Embedding` layer here is **trainable** -- it learns its weights from scratch during training,
unlike the frozen GloVe embeddings you used in TP-04 Part 7.


### 3.1 — Implement `build_rnn_model`


In [ ]:
def build_rnn_model(
    max_vocab: int,
    embed_dim: int,
    max_len: int,
    rnn_units: int = 32,
    learning_rate: float = 1e-3,
) -> tf.keras.Model:
    """
    Build a SimpleRNN text classifier.

    Architecture
    ------------
    Embedding(max_vocab, embed_dim, input_length=max_len)
    SimpleRNN(rnn_units)
    Dense(1, activation='sigmoid')

    Parameters
    ----------
    max_vocab : int
        Vocabulary size (input dimension of Embedding layer).
    embed_dim : int
        Embedding dimension.
    max_len : int
        Input sequence length.
    rnn_units : int, optional
        Number of hidden units in the SimpleRNN layer, by default 32.
    learning_rate : float, optional
        Adam learning rate, by default 1e-3.

    Returns
    -------
    tf.keras.Model
        Compiled model ready to fit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
rnn_model = build_rnn_model(MAX_VOCAB, EMBED_DIM, MAX_LEN, rnn_units=32, learning_rate=LR)
rnn_model.summary()


In [ ]:
history_rnn = rnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)
print(f'Final val accuracy: {history_rnn.history["val_accuracy"][-1]:.4f}')


---
## Part 4 — LSTM Classifier

An LSTM replaces `SimpleRNN` with a gated unit that explicitly controls what to
remember and what to forget. It has three gates:

- **Forget gate** $f_t$: how much of the previous cell state to keep
- **Input gate** $i_t$: how much of the new input to write into the cell
- **Output gate** $o_t$: how much of the cell state to expose as hidden state

The key addition is the **cell state** $c_t$ — a separate memory channel that
passes through the sequence with only gentle, controlled modifications.
This is what allows LSTMs to remember information from hundreds of steps ago
without the gradients vanishing.


### 4.1 — Implement `build_lstm_model`


In [ ]:
def build_lstm_model(
    max_vocab: int,
    embed_dim: int,
    max_len: int,
    lstm_units: int = 32,
    dropout: float = 0.3,
    learning_rate: float = 1e-3,
) -> tf.keras.Model:
    """
    Build an LSTM text classifier.

    Architecture
    ------------
    Embedding(max_vocab, embed_dim, input_length=max_len)
    LSTM(lstm_units)
    Dropout(dropout)
    Dense(1, activation='sigmoid')

    Parameters
    ----------
    max_vocab : int
        Vocabulary size.
    embed_dim : int
        Embedding dimension.
    max_len : int
        Input sequence length.
    lstm_units : int, optional
        Number of LSTM hidden units, by default 32.
    dropout : float, optional
        Dropout rate after LSTM layer, by default 0.3.
    learning_rate : float, optional
        Adam learning rate, by default 1e-3.

    Returns
    -------
    tf.keras.Model
        Compiled model ready to fit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
lstm_model = build_lstm_model(MAX_VOCAB, EMBED_DIM, MAX_LEN, lstm_units=32, learning_rate=LR)
lstm_model.summary()


In [ ]:
history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)
print(f'Final val accuracy: {history_lstm.history["val_accuracy"][-1]:.4f}')


---
## Part 5 — Compare and Analyse


### 5.1 — Implement `plot_training_curves`


In [ ]:
def plot_training_curves(
    histories: dict[str, tf.keras.callbacks.History],
    metric: str = 'accuracy',
) -> plt.Figure:
    """
    Plot training and validation curves for multiple models.

    Parameters
    ----------
    histories : dict of str → History
        Mapping from model name to its Keras History object.
    metric : str, optional
        Metric to plot ('accuracy' or 'loss'), by default 'accuracy'.

    Returns
    -------
    matplotlib.figure.Figure
    """
    # YOUR CODE HERE
    raise NotImplementedError


fig = plot_training_curves(
    {'SimpleRNN': history_rnn, 'LSTM': history_lstm},
    metric='accuracy',
)
plt.show()

fig2 = plot_training_curves(
    {'SimpleRNN': history_rnn, 'LSTM': history_lstm},
    metric='loss',
)
plt.show()


### 5.2 — Evaluate on the test set


In [ ]:
for name, model in [('SimpleRNN', rnn_model), ('LSTM', lstm_model)]:
    y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    print(f'\n=== {name} — test set ===')
    print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=['ham', 'spam']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{name} — test set')
    plt.tight_layout()
    plt.show()


**📝 Your analysis — Part 5**

1. Which model converges faster? Which is more stable?
2. Do you see signs of overfitting (train accuracy >> val accuracy)? At which epoch does it start?
3. Compare precision and recall for the spam class. Which matters more for a spam filter — missing a spam (false negative) or flagging a ham (false positive)? Does your model reflect this preference?
4. Look at the confusion matrix. How many legitimate messages are incorrectly flagged as spam?

*Write your answers here.*


---
## Part 6 (Bonus) — Dense Baseline: Does Word Order Help?

Build a model that uses the same `Embedding` layer but replaces the RNN with a `GlobalAveragePooling1D`
layer (equivalent to mean pooling). This model ignores word order entirely.

Compare its performance with SimpleRNN and LSTM. If the Dense baseline performs comparably,
word order is not very useful for this particular task.


In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling1D


def build_dense_baseline(
    max_vocab: int,
    embed_dim: int,
    max_len: int,
    learning_rate: float = 1e-3,
) -> tf.keras.Model:
    """
    Build a mean-pooling text classifier (no recurrence, no word order).

    Architecture
    ------------
    Embedding(max_vocab, embed_dim, input_length=max_len)
    GlobalAveragePooling1D()   ← mean over the sequence dimension
    Dense(1, activation='sigmoid')

    Parameters
    ----------
    max_vocab : int
    embed_dim : int
    max_len : int
    learning_rate : float, optional

    Returns
    -------
    tf.keras.Model
        Compiled model.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
dense_model = build_dense_baseline(MAX_VOCAB, EMBED_DIM, MAX_LEN, LR)
history_dense = dense_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)

fig = plot_training_curves(
    {'SimpleRNN': history_rnn, 'LSTM': history_lstm, 'Dense (no order)': history_dense},
    metric='accuracy',
)
plt.show()


**📝 Your analysis — Bonus**

1. Does the Dense baseline perform comparably to SimpleRNN and LSTM?
2. What does this tell you about how important word order is for spam detection?
3. Can you think of an NLP task where word order would matter *much* more than it does here?

*Write your answers here.*


---
## Summary

Run the cell below to print your final results.


In [ ]:
print('TP 05-A — Spam detection results')
print(f'Dataset : {len(df)} messages ({(df.label==0).sum()} ham, {(df.label==1).sum()} spam)')
print(f'MAX_LEN : {MAX_LEN}   EMBED_DIM : {EMBED_DIM}   EPOCHS : {EPOCHS}')
print()
for name, model in [('SimpleRNN', rnn_model), ('LSTM', lstm_model)]:
    y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    from sklearn.metrics import f1_score
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f'{name:<20} Test Macro F1: {f1:.4f}')
print()
print('What is next: TP 05-B — Emojifier with GloVe frozen embeddings + LSTM')
